In [0]:
dbutils.widgets.dropdown(name = "environment", defaultValue= "dev",choices= ["dev","prod","qa"],label = "select Environment")

In [0]:
env = dbutils.widgets.get("environment")
print(env)

#saleslake_prd.bronze_prd.rawCustomer
brznTablName = f"saleslake_{env}.bronze_{env}.rawinvoice"
print(brznTablName)
srcFileLoc=f"s3://saleslakekir/saleslake/src_file/DatabricksSourceFile/invoice.csv"
print(srcFileLoc)

In [0]:
from pyspark.sql import functions as F

# Read the source CSV file
df = (
    spark.read
         .option("header", "true")
         .csv(srcFileLoc)
)

# Apply transformations (add ingest timestamp, rename columns if needed)
df_transformed = (
    df.withColumn("ingest_ts", F.current_timestamp())
)

# Write into the target Delta table
(
    df_transformed.write
                  .format("delta")   # Delta Lake for ACID guarantees
                  .mode("append")    # or "overwrite" depending on your use case
                  .option("mergeSchema", "false")  # same as COPY_OPTIONS
                  .saveAsTable(brznTablName)      # target table name
)



In [0]:
%sql
SELECT * FROM saleslake_qa.bronze_qa.rawInvoice;

In [0]:
%sql
select count(*) from saleslake_prod.bronze_prod.rawInvoice ;